# Convert BAM files to bigWig (bigwig) tracks


## Aim

This module converts coordinate-sorted BAM alignment files into bigWig (`.bw`) coverage tracks, one per input BAM. Each BAM is indexed with `samtools index` and converted to a normalized coverage track with deepTools `bamCoverage`. bigWig tracks are compact, indexed, and can be loaded directly into genome browsers (e.g. IGV, UCSC) for visual inspection of read coverage.


## Input

- One or more coordinate-sorted BAM files (e.g. `*.Aligned.sortedByCoord.out.bam`)

> **Note (MWE input):** these coordinate-sorted BAMs are produced by the RNA-seq alignment step (`RNA_calling`, STAR). In this toy example they are pre-staged under `output/rnaseq/intermediates/bam/`, so run `RNA_calling` first (or use the pre-staged toy BAMs) before this step.

## Output

- One bigWig coverage track (`.bw`) per input BAM, written to the working directory


## Minimal working example

**Step 1.** Convert the toy RNA-seq BAM files to bigWig coverage tracks:


**Timing:** Runtime varies by dataset size and compute resources. For the toy chr22 MWE dataset, most steps complete in under 10 minutes on a standard HPC node.

In [ ]:
sos run pipeline/bam_to_bw.ipynb reformat_vcf_wasp \
    --cwd output/bam_to_bw \
    --bam_files `ls output/rnaseq/intermediates/bam/SAMPLE_00*.Aligned.sortedByCoord.out.bam`


## Command interface


In [ ]:
sos run pipeline/bam_to_bw.ipynb -h


In [ ]:
[global]
import os
# Work directory & output directory
parameter: cwd = path('./')
# The filename prefix for output data
parameter: job_size = 1
parameter: mem = '60G'
parameter: container = ""
parameter: bam_files = paths
import pandas as pd
#parameter: analysis_units = path
# handle N = per_chunk data-set in one job
parameter: per_chunk = 1


## Workflow implementation


## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[reformat_vcf_wasp]
input: bam_files, group_by = 1
output:f'{cwd:a}/{_input:bn}.bw'
task: trunk_workers = 1, trunk_size = job_size, mem = mem,  walltime = '24h', tags = f'{step_name}_{_output[0]:bn}'
bash: expand= "$[ ]", stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout', container=container
    samtools index $[_input]
    bamCoverage -b $[_input] -o $[_output]
